In [265]:
import Utils.ab_makeData as abn
import peppi_py as pp


import torch
import torch.nn as nn
import torch.nn.functional as F
from matplotlib.pyplot import plot, xlim, ylim
import numpy


##
import pandas as pd
import os
from datetime import datetime
import pickle as pkl

## Model

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Data

In [229]:
## Model
DATA_FOLDER = r"C:\Users\soph\PVV\Projects\Smash\shared_raw"
gameName = r'\02_32_35 [RUDE] Marth + [INFP] Marth (BF).slp'
test_path = DATA_FOLDER + r"\Slippi_Public_Dataset_v3" + gameName
inputFolder = DATA_FOLDER+ r"\Slippi_Public_Dataset_v3"
###


In [59]:
class TestClass():
    def __init__(self, chec):
        self.start = chec
        self.double = self.double(self.start)
        self.implciti = self.implicitDouble()

    def double(self, double:int):
        return 2*double
    
    def implicitDouble(self):
        return 3*self.start
    
TestClass(2).implciti

6

device(type='cuda', index=0)

In [291]:
inputColumns = ['x_pos', 'y_pos', 'x_pos.enemy', 'y_pos.enemy']
outputColumns = ['x_joy', 'y_joy']
enemySplitList = ['x_pos', 'y_pos', 'x_joy', 'y_joy', 'frameNumber', 'character']


def inputStart(dfGood, dfBad):
   return dfGood.merge(dfBad.add_suffix(".enemy"), left_on='frameNumber', right_on='frameNumber.enemy')

def customDFtoTensor(inputPath : str):
    
    inputDF = pd.DataFrame(abn.make_pre_large(pp.read_slippi(inputPath)))

    
    Player0 = inputDF[inputDF['Player']=='0'][enemySplitList]
    Player1 = inputDF[inputDF['Player']=='1'][enemySplitList]

    preFinal = pd.concat([inputStart(Player0, Player1), inputStart(Player1, Player0)])
    preFinal['stage'] = inputDF['stage'][0]
    preFinal = preFinal.astype('float32')

    return preFinal


class GamesDataSet(torch.utils.data.Dataset):
    def __init__(self, image_folder, characterFilter = 'Marth'):
        self.games_folder = image_folder
        self.characterFilter = characterFilter
        self.games = self.filterGames()

    ## part of init
    def filterGames(self):
        games = []
        for filename in os.listdir(self.games_folder):
            if self.characterFilter in filename:
                games.append(filename)
        print("total games: " + str(len(games)))
        return games
    
    def __getitem__(self, index):
        try:
            pathToGame = self.games_folder + r"\\" + self.games[index]
            if str(pathToGame).endswith('.pkl'):
                majority_df = pd.read_pickle(pathToGame)
                print(majority_df)
            else:
                almonstFinal = customDFtoTensor(pathToGame)
                majority_df = almonstFinal[almonstFinal['character']==9.0]

            inputTensor = torch.tensor(majority_df[inputColumns].values)
            targetTensor = torch.tensor(majority_df[outputColumns].values)
        except:
            return None
            ## details hanlded in custon collate_fn

        return inputTensor, targetTensor
    
    def __len__(self):
        return len(self.games)
        

## concatenating together for ease
def collate_fn(batch):
    batch = list(filter(lambda x: x is not None, batch))

    data = [item[0] for item in batch]
    data = torch.cat(data, dim=0)
    target = [item[1] for item in batch]
    target = torch.cat(target, dim=0)

    return [data, target]
    #return torch.utils.data.dataloader.default_collate(batch)

def filterGames(games_folder,characterFilter):
    games = []
    for filename in os.listdir(games_folder):
        if characterFilter in filename:
            games.append(filename)
    print("total games: " + str(len(games)))
    return games

def cleanSLPs(inputPath:str):
    cleanedPrefix = r"\\cleaned"
    if not os.path.isdir(inputPath + cleanedPrefix):
        print("doesn't exist, making")
        os.mkdir(inputPath + cleanedPrefix)

    games_list = filterGames(inputPath, 'Marth')
    start = datetime.now()
    count = 0
    for game in games_list:
        gameFullPaths = inputPath + game
        almonstFinal = customDFtoTensor(gameFullPaths)
        majority_df = almonstFinal[almonstFinal['character']==9.0]
        ## new filepath
        game_prefix = game[:-4]
        pd.to_pickle(majority_df, inputPath + cleanedPrefix + r"\\" + game_prefix + ".pkl")

    print("Done in: " + str(datetime.now() - start))
        
        




In [292]:
cleanSLPs(inputFolder)

total games: 15699
25314
18414
14422
19930
17678
24512
30064
14104
24000
23640
26542
28400
15098
28022
17018
20958
15538
16584
23604
22538
17554
23656
14622
22806
17642
17224
16864
21216
9032
19036
8186
26476
20208
27562
18248
17130
19768
18186
14844
10170
18758
13656
21380
15398
18902
31434
13006
18834
26762
21340
15088
15272
16374
14110
11768
18092
20756
12352
25714
28546
19002
13272
18466
10734
15230
21800
15082
21946
22322
19294
27986
19084
24202
14666
20642
22506
26090
22842
14986
10606
18912
19112
19182
11506
18760
18152
17596
18850
28342
20908
13982
21646
15926
21960
16564
14886
20208
14216
10452
17636
16042
21696
17434
16690
22486
19102
13708
28454
14904
23758
21668
17448
9390
18280
16576
13938
10568
20804
14164
16866
20218
18356
13954
17586
28252
13812
11966
11538
18552
22802
25294
14338
17016
25840
17078
10770
18294
22084
19114
20772
22528
20606
21192
19102
17392
6850
17734
19660
13738
17342
14930
17924
18460
27014
21292
16298
14920
16078
19496
12376
16252
5316
15770
19196
22

PanicException: assertion `left == right` failed
  left: 10443
 right: 8398

In [293]:
os.listdir(inputFolder+r"\\cleaned")

['00_46_05.192Z Marth + [MS] Fox (BF).pkl',
 '00_46_06 [RUDE] Marth + Fox (BF).pkl',
 '00_48_53 [RUDE] Marth + Fox (YI).pkl',
 '00_49_47.618Z Marth + [MS] Fox (DL).pkl',
 '00_51_07 [RUDE] Marth + Fox (FD).pkl',
 '00_52_58.988Z Marth + [MS] Fox (DL).pkl',
 '00_53_47 [RUDE] Marth + Fox (DL).pkl',
 '00_56_35.789Z Marth + [MS] Fox (FD).pkl',
 '01_02_49.857Z Jigglypuff + Marth (BF).pkl',
 '01_06_23.913Z Jigglypuff + Marth (PS).pkl',
 '01_08_09.539Z [EASY] Fox + [=P] Marth (BF).pkl',
 '01_09_54.529Z Jigglypuff + Marth (DL).pkl',
 '01_12_03.932Z [EASY] Fox + [=P] Marth (FD).pkl',
 '01_14_04.170Z Jigglypuff + Marth (YI).pkl',
 '01_14_37.489Z [EASY] Fox + [=P] Marth (YS).pkl',
 '01_15_46.571Z Fox + Marth (FoD).pkl',
 '01_17_23.016Z [EASY] Fox + [=P] Marth (YS).pkl',
 '01_17_50 Marth + Captain Falcon (YI).pkl',
 '01_18_35.481Z Jigglypuff + Marth (FoD).pkl',
 '01_19_47.530Z [EASY] Fox + [=P] Marth (PS).pkl',
 '01_24_53 Captain Falcon + [RUDE] Marth (BF).pkl',
 '01_25_27 [PPAP] Marth + Fox (BF).pk

In [290]:
inputFolder = DATA_FOLDER+ r"\Slippi_Public_Dataset_v3\\"
pickleInputFolder = inputFolder+r"\\cleaned"
marthGames =  GamesDataSet(inputFolder)
marthGames.__getitem__(2)


total games: 15699
14422


(tensor([[-42.0000,  26.6000,  42.0000,  28.0000],
         [-42.0000,  26.6000,  42.0000,  28.0000],
         [-42.0000,  26.6000,  42.0000,  28.0000],
         ...,
         [-56.6685, -83.8778, -59.8400,  -7.3400],
         [-56.6685, -86.3778, -58.9511,  -5.4911],
         [-56.6685, -88.8778, -57.2089,  -4.0689]]),
 tensor([[0., 0.],
         [0., 0.],
         [0., 0.],
         ...,
         [0., 0.],
         [0., 0.],
         [0., 0.]]))

In [212]:
test_size = .15
validation_size = .15
batch_size = 32 ## optimze later

test_amount, val_amount = int(marthGames.__len__() * test_size), int(marthGames.__len__() * validation_size)

train_set, val_set, test_set = torch.utils.data.random_split(marthGames, [
            (marthGames.__len__() - (test_amount + val_amount)), 
            test_amount, 
            val_amount
])

train_dataloader = torch.utils.data.DataLoader(
            train_set,
            batch_size=batch_size,
            shuffle=True,
            collate_fn=collate_fn
)
val_dataloader = torch.utils.data.DataLoader(
            val_set,
            batch_size=batch_size,
            shuffle=True,
            collate_fn=collate_fn
)
test_dataloader = torch.utils.data.DataLoader(
            test_set,
            batch_size=batch_size,
            shuffle=True,
            collate_fn=collate_fn
)

### trial debug

debug_dataloader = torch.utils.data.DataLoader(
            torch.utils.data.Subset(marthGames, [0,3]),
            batch_size=2,
            shuffle=True,
            collate_fn=collate_fn
)

train_dataloader2 = torch.utils.data.DataLoader(
            train_set,
            batch_size=2,
            shuffle=True,
            collate_fn=collate_fn,
)


In [176]:
a1 = marthGames.__getitem__(2)
a2 = marthGames.__getitem__(3)

C:\Users\soph\PVV\Projects\Smash\shared_raw\Slippi_Public_Dataset_v3\\00_48_53 [RUDE] Marth + Fox (YI).slp
14422
C:\Users\soph\PVV\Projects\Smash\shared_raw\Slippi_Public_Dataset_v3\\00_49_47.618Z Marth + [MS] Fox (DL).slp
19930


In [214]:
for a,b in train_dataloader2:
    print('hi')

C:\Users\soph\PVV\Projects\Smash\shared_raw\Slippi_Public_Dataset_v3\\20200129 - HNC 4 - PM 0651 - Marth (Black) vs Ganondorf (Default) - Yoshi_s Story.slp
16682
C:\Users\soph\PVV\Projects\Smash\shared_raw\Slippi_Public_Dataset_v3\\20_22_35 Fox + Marth (FoD).slp
11430
torch.Size([14056, 4])
torch.Size([14056, 2])
hi
C:\Users\soph\PVV\Projects\Smash\shared_raw\Slippi_Public_Dataset_v3\\13_06_10 [C2] Marth + Fox (DL).slp
25872
C:\Users\soph\PVV\Projects\Smash\shared_raw\Slippi_Public_Dataset_v3\\17_44_53 Falco + Marth (PS).slp


KeyboardInterrupt: 

In [ ]:
start = datetime.now()
for _ in range(10):
    current = datetime.now()
    print(current-start)

0:00:08.264610
0:00:08.264610
0:00:08.264610
0:00:08.264610
0:00:08.264610
0:00:08.264610
0:00:08.264610
0:00:08.264610
0:00:08.264610
0:00:08.264610


In [ ]:
torch.cat((a1[0], a2[0], tensor1),0)


tensor1 = torch.randn(7211, 4)  # Tensor with shape [7211, 4]
tensor2 = torch.randn(9965, 4)  # Tensor with shape [9965, 4]
print(a1[0].size())
print(a1[1].size())

print(a2[0].size())
print(a2[1].size())

tensor([[-42.0000,  26.6000,  42.0000,  28.0000],
        [-42.0000,  26.6000,  42.0000,  28.0000],
        [-42.0000,  26.6000,  42.0000,  28.0000],
        ...,
        [ -1.5428,   0.3055,   0.9177,  -1.1384],
        [ -1.5927,   1.9690,   1.8775,  -0.3367],
        [ -1.5457,  -0.3197,  -0.7822,  -0.3895]])

## Train

In [ ]:

# Concatenate tensors along dimension 0
result = torch.cat((tensor1, tensor2), dim=0)

print(result.shape)

torch.Size([17176, 4])


In [100]:
class Net(nn.Module):
    def __init__(self, input_size, output_size, layerSize):
        super(Net,self).__init__()
        self.fc1 = nn.Linear(input_size, layerSize)
        self.fc2 = nn.Linear(layerSize,layerSize)
        self.fc3 = nn.Linear(layerSize,layerSize)
        self.fc4 = nn.Linear(layerSize,layerSize)
        self.fc5 = nn.Linear(layerSize,layerSize)
        self.fc6 = nn.Linear(layerSize, output_size)

    def forward(self, x):
        x = F.sigmoid(self.fc1(x))
        x = F.sigmoid(self.fc2(x))
        x = F.sigmoid(self.fc3(x))
        x = F.sigmoid(self.fc4(x))
        x = F.sigmoid(self.fc5(x))
        x = self.fc6(x)
        return x
    
Smash1 = Net(4,2,32).cuda()

In [ ]:
# Smash1.to(device)

Net(
  (fc1): Linear(in_features=4, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=32, bias=True)
  (fc4): Linear(in_features=32, out_features=32, bias=True)
  (fc5): Linear(in_features=32, out_features=32, bias=True)
  (fc6): Linear(in_features=32, out_features=2, bias=True)
)

In [ ]:
## manually dumb train

EPOCHS = 1000

loss_list = []
total_count = 0
for index in range(EPOCHS):

    for inputTensorBegin, targetTensorBeing in test_dataloader:
        print('something')
        inputTensor = inputTensorBegin.cuda()
        targetTensor = targetTensorBeing.cuda()

        currentTensor = Smash1(inputTensor)
        lossFunction = nn.MSELoss()
        loss = lossFunction(currentTensor, targetTensor)
        loss_list.append(loss)
        loss.backward()
        for p in Smash1.parameters():
            p.data.add_(-0.001 * p.grad)
            p.grad.data.zero_()
        total_count = total_count + 1

        print(total_count)
        if total_count % 300 == 0 :
            print(index)
            print(p.grad)
            torch.save(Smash1, "./" + str(total_count) + "_Model3.pt")


In [ ]:
Smash1.train()

Net(
  (fc1): Linear(in_features=4, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=32, bias=True)
  (fc4): Linear(in_features=32, out_features=32, bias=True)
  (fc5): Linear(in_features=32, out_features=32, bias=True)
  (fc6): Linear(in_features=32, out_features=2, bias=True)
)

In [18]:
savePath = "Model2.pt"
torch.save(Smash1.state_dict(), savePath)


In [19]:
newGuy = torch.load("Model1.pt", weights_only=True)
Smash2 = Net(4,2,32)
Smash2.load_state_dict(newGuy)
Smash2.eval()



Net(
  (fc1): Linear(in_features=4, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=32, bias=True)
  (fc4): Linear(in_features=32, out_features=32, bias=True)
  (fc5): Linear(in_features=32, out_features=32, bias=True)
  (fc6): Linear(in_features=32, out_features=2, bias=True)
)

In [20]:
import numpy as np
test_input = torch.tensor([1,2,3,4]).to(dtype=torch.float32)
Smash2(test_input).detach().numpy()

array([-0.02702457, -0.13033545], dtype=float32)

In [21]:
torch.accelerator.is_available()

True

In [ ]:
import re

text = "[tensor(0.3466, device='cuda:0', grad_fn=<MseLossBackward0>)"

# Use regex to extract values after tensor(
matches = re.findall(r'(?<=tensor\()([^\)]+)', text)

with open( "../loss_list.txt ", 'r') as text:
   for x in text:
      text = x
text
matches = re.findall(r'(?<=tensor\()([^\)]+)', text)
total=[]
for match in matches:
   print(match)
   total.append(re.findall(r'(?<=tensor\()([0-9.]+|nan)', match))



0.3466, device='cuda:0', grad_fn=<MseLossBackward0>
nan, device='cuda:0', grad_fn=<MseLossBackward0>
nan, device='cuda:0', grad_fn=<MseLossBackward0>
nan, device='cuda:0', grad_fn=<MseLossBackward0>
nan, device='cuda:0', grad_fn=<MseLossBackward0>
nan, device='cuda:0', grad_fn=<MseLossBackward0>
nan, device='cuda:0', grad_fn=<MseLossBackward0>
nan, device='cuda:0', grad_fn=<MseLossBackward0>
nan, device='cuda:0', grad_fn=<MseLossBackward0>
nan, device='cuda:0', grad_fn=<MseLossBackward0>
nan, device='cuda:0', grad_fn=<MseLossBackward0>
nan, device='cuda:0', grad_fn=<MseLossBackward0>
nan, device='cuda:0', grad_fn=<MseLossBackward0>
nan, device='cuda:0', grad_fn=<MseLossBackward0>
nan, device='cuda:0', grad_fn=<MseLossBackward0>
nan, device='cuda:0', grad_fn=<MseLossBackward0>
nan, device='cuda:0', grad_fn=<MseLossBackward0>
nan, device='cuda:0', grad_fn=<MseLossBackward0>
nan, device='cuda:0', grad_fn=<MseLossBackward0>
nan, device='cuda:0', grad_fn=<MseLossBackward0>
nan, device='cuda